# Pairs Trading

Same pattern as every other notebook, plus one strategy-specific chart: the spread z-score with its entry/exit bands.

This strategy is market-neutral (long one leg, short the other), so its equity curve isn't directly comparable to a directional benchmark like SPY leg-for-leg. It's still useful, though, as the rawest possible check: given the same starting capital, did this strategy beat just buying and holding the benchmark?

In [1]:
from data import get_prices
from algorithm import PairsTrading
from backtest import Backtester, build_report
from backtest.report import pairs_spread_chart

In [2]:
from datetime import date

TICKER_A, TICKER_B, BENCHMARK = "KO", "PEP", "SPY"
START, END = "2018-01-01", date.today().isoformat()  # through today

prices = get_prices([TICKER_A, TICKER_B], START, END)
benchmark = get_prices([BENCHMARK], START, END)[BENCHMARK]
prices.tail()

,KO,PEP
Date,,
2026-06-24,80.599998,142.270004
2026-06-25,80.419998,139.520004
2026-06-26,82.629997,141.389999
2026-06-29,82.650002,138.679993
2026-06-30,81.269997,135.399994


In [3]:
algorithm = PairsTrading(TICKER_A, TICKER_B, window=60, entry_z=2.0, exit_z=0.5)
result = Backtester().run(algorithm, prices, initial_capital=100_000, commission_bps=5, slippage_bps=5)
result.equity.tail()

Date
2026-06-24    111350.611142
2026-06-25    110398.775596
2026-06-26    109621.697686
2026-06-29    108557.874709
2026-06-30    108180.386904
dtype: float64

In [4]:
report = build_report(result, benchmark_prices=benchmark)
report.stats

{'total_return': np.float64(0.08180386903501624),
 'cagr': np.float64(0.00932849664516544),
 'annual_volatility': np.float64(0.04727402032180862),
 'sharpe': 0.22004422502245327,
 'sortino': 0.1917938820639846,
 'max_drawdown': -0.13369348375178236,
 'calmar': np.float64(0.06977525293966376),
 'win_rate': 0.5005740528128588,
 'trade_count': 55,
 'benchmark_total_return': np.float64(2.1648934261118344),
 'benchmark_cagr': np.float64(0.1457409961792573)}

In [5]:
for fig in report.figures:
    fig.show()

In [6]:
# Strategy-specific chart: the spread z-score the algorithm trades on.
spread, zscore = algorithm.compute_spread_zscore(prices)
pairs_spread_chart(spread, zscore, algorithm.entry_z, algorithm.exit_z).show()